In [ ]:
# ── Colab Setup ──────────────────────────────────────────────────────────────
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/MScProject'
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
    os.chdir(PROJECT_ROOT)
    print(f"Working directory: {os.getcwd()}")
else:
    src_dir = os.path.join(os.getcwd(), 'src')
    if os.path.isdir(src_dir) and src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    print("Running locally.")

# 04 — Attention Head Analysis

Investigates which attention heads in a trained **Transformer** language model
correspond to grammatical rules of the artificial language.

Barry's core question: *do particular attention heads correspond to particular rules of the grammar?*

For **aⁿbⁿ** the key structural dependency is the cross-serial link: the model must
count n 'a' tokens and then produce exactly n 'b' tokens.  We look for heads where
'b'-phase positions attend strongly back to 'a'-phase positions — a head implementing
this pattern is a good candidate for encoding the counting rule.

**Analyses:**
1. **Attention heatmaps** — qualitative visual inspection per head and layer
2. **Cross-phase attention score** — quantitative: how much do b-tokens attend to a-tokens?
3. **Attention entropy** — does a head spread attention diffusely or focus sharply?

Requires a **Transformer** checkpoint from `02_train.ipynb` (`MODEL_TYPE = 'transformer'`).

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from torch.utils.data import DataLoader, Dataset

from grammar_loader import load_grammar, build_vocab, tokenize


# ── Model classes (self-contained copy) ─────────────────────────────────────

class GrammarDataset(Dataset):
    def __init__(self, filepath, vocab):
        self.sequences = []
        with open(filepath) as f:
            for line in f:
                line = line.strip()
                if line:
                    ids = tokenize(line, vocab, add_special=True)
                    self.sequences.append(torch.tensor(ids, dtype=torch.long))

    def __len__(self):  return len(self.sequences)
    def __getitem__(self, idx): return self.sequences[idx]


def collate_fn(batch, pad_id):
    max_len = max(s.size(0) for s in batch)
    out = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    for i, s in enumerate(batch):
        out[i, :s.size(0)] = s
    return out


class TransformerLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_heads=4, num_layers=2,
                 ff_dim=256, dropout=0.1, max_seq_len=512, causal_mask=True):
        super().__init__()
        self.causal_mask  = causal_mask
        self.embedding    = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_seq_len, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True
        )
        self.transformer  = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_proj  = nn.Linear(embed_dim, vocab_size)
        self.num_layers   = num_layers

    def forward(self, x):
        B, T = x.shape
        pos  = torch.arange(T, device=x.device).unsqueeze(0)
        emb  = self.embedding(x) + self.pos_embedding(pos)
        pad_mask = (x == 0)
        causal   = None
        if self.causal_mask:
            causal = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        out = self.transformer(emb, mask=causal, src_key_padding_mask=pad_mask)
        return self.output_proj(out)

## Configuration

Point `CHECKPOINT` at a **Transformer** checkpoint saved by `02_train.ipynb`.
Run notebook 02 with `MODEL_TYPE = 'transformer'` first if you haven't already.

`EXAMPLE_N` controls which sequence lengths are used for the heatmap visualisations.

In [ ]:
CHECKPOINT   = '/content/drive/My Drive/Colab Notebooks/MScProjectCode/checkpoints/anbn_transformer.pt'
GRAMMAR_FILE = 'grammars/anbn.txt'
DATA_FILE    = 'data/anbn_n1-100.txt'
FIGURES_DIR  = 'figures'

EXAMPLE_N    = [5, 10, 20]   # n values to visualise attention heatmaps for
BATCH_SIZE   = 64

## Load checkpoint

In [ ]:
device = torch.device('cpu')

ckpt = torch.load(CHECKPOINT, map_location=device)
assert ckpt['model_type'] == 'transformer', \
    f"This notebook requires a transformer checkpoint, got '{ckpt['model_type']}'. "\
    "Re-run 02_train.ipynb with MODEL_TYPE = 'transformer'."

vocab      = ckpt['vocab']
model_args = ckpt['args']
pad_id     = vocab['[PAD]']
id_to_tok  = {v: k for k, v in vocab.items()}

model = TransformerLanguageModel(
    vocab_size  = len(vocab),
    embed_dim   = model_args['embed_dim'],
    num_heads   = 4,
    num_layers  = model_args['num_layers'],
    ff_dim      = model_args['hidden_dim'],
    causal_mask = ckpt['causal_mask'],
)
model.load_state_dict(ckpt['model_state'])
model.eval()

num_layers = model_args['num_layers']
num_heads  = 4
print(f"Loaded: {ckpt['grammar_name']} / transformer  "
      f"({num_layers} layers, {num_heads} heads)")

## Attention weight extraction

PyTorch's `TransformerEncoderLayer` discards attention weights internally.
We recover them by manually running each layer's `self_attn` with
`need_weights=True` (at the same inputs the layer will use), then advancing
through the full layer normally.

In [ ]:
@torch.no_grad()
def get_attention_weights(model, batch):
    """
    Run the transformer on `batch` and return per-layer attention weights.

    Returns
    -------
    list of length num_layers, each tensor (B, num_heads, T, T)
    """
    B, T = batch.shape
    pos  = torch.arange(T, device=batch.device).unsqueeze(0)
    x    = model.embedding(batch) + model.pos_embedding(pos)
    pad_mask = (batch == 0)
    causal   = None
    if model.causal_mask:
        causal = nn.Transformer.generate_square_subsequent_mask(T, device=batch.device)

    attn_per_layer = []
    for layer in model.transformer.layers:
        # Capture weights at this layer's input
        _, weights = layer.self_attn(
            x, x, x,
            attn_mask            = causal,
            key_padding_mask     = pad_mask,
            need_weights         = True,
            average_attn_weights = False,   # keep per-head: (B, H, T, T)
        )
        attn_per_layer.append(weights.detach().cpu())   # (B, num_heads, T, T)
        # Advance x through this layer
        x = layer(x, src_mask=causal, src_key_padding_mask=pad_mask)

    return attn_per_layer


def ids_to_tokens(ids, id_to_tok):
    """Convert a 1-D tensor of token IDs to a list of string labels."""
    return [id_to_tok.get(i.item(), '?') for i in ids]


def seq_for_n(n, vocab):
    """Build a single aⁿbⁿ sequence as a (1, T) tensor."""
    s   = 'a' * n + 'b' * n
    ids = tokenize(s, vocab, add_special=True)
    return torch.tensor(ids, dtype=torch.long).unsqueeze(0)

## Attention heatmaps — example sequences

For each n in `EXAMPLE_N`, plot a grid of heatmaps: rows = layers, columns = heads.

**What to look for:**
- Any head where 'b' tokens (right half) have high attention to 'a' tokens (left half)
  is a candidate for encoding the counting dependency.
- Diagonal patterns → local/positional attention.
- Columns all lit up → a 'sink' token that everything attends to.

In [ ]:
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

for n in EXAMPLE_N:
    batch  = seq_for_n(n, vocab).to(device)
    tokens = ids_to_tokens(batch[0], id_to_tok)
    attn   = get_attention_weights(model, batch)   # list of (1, H, T, T)

    fig, axes = plt.subplots(
        num_layers, num_heads,
        figsize=(num_heads * 3.2, num_layers * 3.0),
    )
    if num_layers == 1:
        axes = [axes]

    for l in range(num_layers):
        for h in range(num_heads):
            ax  = axes[l][h]
            mat = attn[l][0, h].numpy()     # (T, T)
            im  = ax.imshow(mat, aspect='auto', cmap='Blues', vmin=0, vmax=mat.max())
            ax.set_title(f"L{l+1} H{h+1}", fontsize=9)
            ax.set_xticks(range(len(tokens)))
            ax.set_yticks(range(len(tokens)))
            ax.set_xticklabels(tokens, rotation=90, fontsize=7)
            ax.set_yticklabels(tokens, fontsize=7)
            if h == 0:
                ax.set_ylabel('Query', fontsize=8)
            if l == num_layers - 1:
                ax.set_xlabel('Key', fontsize=8)

    fig.suptitle(f"{ckpt['grammar_name']}  n={n}  — attention weights", fontsize=11)
    fig.tight_layout()
    path = f"{FIGURES_DIR}/{ckpt['grammar_name']}_transformer_attn_n{n}.png"
    fig.savefig(path, dpi=150)
    plt.show()
    print(f"Saved: {path}")

## Cross-phase attention

Quantifies how much each head attends **across** the a→b boundary.

For every sequence, for every b-position query, we compute the total attention
weight directed to a-position keys.  Averaging over all sequences gives a score
in [0, 1] per head per layer.

A **high score** means that head uses b-positions to look back at a-positions —
exactly the behaviour needed to track the counting dependency.

In [ ]:
grammar = load_grammar(GRAMMAR_FILE)
dataset = GrammarDataset(DATA_FILE, vocab)
dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=lambda b: collate_fn(b, pad_id),
)

a_id = vocab.get('a', -1)
b_id = vocab.get('b', -1)

# Accumulators: (num_layers, num_heads)
cross_sum   = np.zeros((num_layers, num_heads))
cross_count = 0

for batch in dataloader:
    batch = batch.to(device)
    attn  = get_attention_weights(model, batch)   # list of (B, H, T, T)

    for b_idx in range(batch.size(0)):
        toks     = batch[b_idx]                         # (T,)
        a_mask   = (toks == a_id)                       # True at a-positions
        b_mask   = (toks == b_id)                       # True at b-positions
        if not b_mask.any() or not a_mask.any():
            continue

        for l in range(num_layers):
            # attn[l]: (B, H, T, T)  row=query, col=key
            w = attn[l][b_idx]                          # (H, T, T)
            # For b-position queries, sum attention over a-position keys
            b_to_a = w[:, b_mask, :][:, :, a_mask]     # (H, #b, #a)
            cross_sum[l] += b_to_a.sum(dim=(1, 2)).numpy()

        cross_count += b_mask.sum().item()              # number of b-query positions

cross_phase_score = cross_sum / max(cross_count, 1)    # (num_layers, num_heads)

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, num_layers, figsize=(num_layers * 4, 3.5), sharey=True)
if num_layers == 1:
    axes = [axes]

for l, ax in enumerate(axes):
    scores = cross_phase_score[l]
    bars   = ax.bar(range(1, num_heads + 1), scores, color='steelblue')
    ax.set_title(f"Layer {l+1}")
    ax.set_xlabel('Head')
    if l == 0:
        ax.set_ylabel('Mean b→a attention')
    ax.set_ylim(0, max(cross_phase_score.max() * 1.2, 0.1))
    ax.set_xticks(range(1, num_heads + 1))

fig.suptitle(f"{ckpt['grammar_name']} — cross-phase attention (b-tokens attending to a-tokens)",
             fontsize=10)
fig.tight_layout()
path = f"{FIGURES_DIR}/{ckpt['grammar_name']}_transformer_cross_phase.png"
fig.savefig(path, dpi=150)
plt.show()
print(f"Saved: {path}")
print("\nCross-phase scores (layer × head):")
print(np.round(cross_phase_score, 4))

## Attention entropy per head

Shannon entropy of each head's attention distribution (averaged over all query
positions and sequences).

- **High entropy** → head spreads attention broadly (diffuse / positional)
- **Low entropy** → head attends sharply to specific positions (potentially rule-specific)

Heads with low entropy and high cross-phase score are the strongest candidates
for encoding the counting rule.

In [ ]:
entropy_sum   = np.zeros((num_layers, num_heads))
entropy_count = 0

for batch in dataloader:
    batch    = batch.to(device)
    attn     = get_attention_weights(model, batch)
    pad_mask = (batch == pad_id)               # (B, T)

    for b_idx in range(batch.size(0)):
        valid = ~pad_mask[b_idx]               # non-PAD positions
        T_val = valid.sum().item()
        if T_val == 0:
            continue

        for l in range(num_layers):
            w = attn[l][b_idx]                 # (H, T, T)
            # Only non-PAD query rows
            w_valid = w[:, valid, :]           # (H, T_val, T)
            # Clip for numerical stability before log
            w_clip  = w_valid.clamp(min=1e-9)
            H_val   = -(w_clip * w_clip.log()).sum(dim=-1).mean(dim=-1)  # (H,)
            entropy_sum[l]  += H_val.numpy()

        entropy_count += 1

mean_entropy = entropy_sum / max(entropy_count, 1)    # (num_layers, num_heads)

# ── Plot: entropy + cross-phase side by side ─────────────────────────────────
fig, axes = plt.subplots(2, num_layers,
                         figsize=(num_layers * 4, 5),
                         sharey='row')
if num_layers == 1:
    axes = [[axes[0]], [axes[1]]]

for l in range(num_layers):
    ax_e = axes[0][l]
    ax_e.bar(range(1, num_heads + 1), mean_entropy[l], color='darkorange')
    ax_e.set_title(f"Layer {l+1}")
    ax_e.set_xticks(range(1, num_heads + 1))
    if l == 0:
        ax_e.set_ylabel('Mean attention entropy')

    ax_c = axes[1][l]
    ax_c.bar(range(1, num_heads + 1), cross_phase_score[l], color='steelblue')
    ax_c.set_xlabel('Head')
    ax_c.set_xticks(range(1, num_heads + 1))
    if l == 0:
        ax_c.set_ylabel('Cross-phase score')

axes[0][0].set_title("Layer 1 — entropy", fontsize=9)
axes[1][0].set_title("Layer 1 — cross-phase", fontsize=9)
if num_layers > 1:
    for l in range(1, num_layers):
        axes[0][l].set_title(f"Layer {l+1} — entropy", fontsize=9)
        axes[1][l].set_title(f"Layer {l+1} — cross-phase", fontsize=9)

fig.suptitle(f"{ckpt['grammar_name']} — attention entropy (top) and cross-phase score (bottom)",
             fontsize=10)
fig.tight_layout()
path = f"{FIGURES_DIR}/{ckpt['grammar_name']}_transformer_entropy.png"
fig.savefig(path, dpi=150)
plt.show()
print(f"Saved: {path}")
print("\nMean entropy (layer × head):")
print(np.round(mean_entropy, 4))

## Interpretation guide

| Pattern | Interpretation |
|---------|----------------|
| High cross-phase + low entropy | Strong candidate for encoding the counting rule |
| Low cross-phase + low entropy  | Head attends to specific positions but within one phase (e.g. local context) |
| High entropy                   | Diffuse / positional head; less likely to be rule-specific |

Cross-reference with the heatmaps: a head with high cross-phase score should show
a clear off-diagonal block in the heatmap (b-row × a-column lit up).

Repeat this notebook with `CHECKPOINT` pointing at an **anbncn** transformer
checkpoint to compare how a context-sensitive grammar is represented.